In [0]:
from pyspark.sql.functions import col, sum, count, when, current_timestamp

SOURCE_TABLE = "banking.banking.banking_silver_transactions"

KPI_TABLE = "banking.banking.gold_kpi_daily_transactions"
FRAUD_TABLE = "banking.banking.gold_fraud_transactions"
CUSTOMER_TABLE = "banking.banking.gold_customer_summary"

print("🔹 Starting Gold Layer")

In [0]:
if spark.catalog.tableExists(KPI_TABLE):
    print("Incremental load")

    max_time = spark.sql(f"""
        SELECT MAX(silver_processed_time) as max_time 
        FROM {KPI_TABLE}
    """).collect()[0]["max_time"]

    df = spark.table(SOURCE_TABLE).filter(col("silver_processed_time") > max_time)

else:
    print("First run - full load")
    df = spark.table(SOURCE_TABLE)

print(f"Records to process: {df.count()}")

In [0]:
kpi_df = df.groupBy("location") \
    .agg(
        count("*").alias("total_transactions"),
        sum("amount").alias("total_amount"),
        sum(when(col("transaction_type") == "DEBIT", col("amount")).otherwise(0)).alias("total_debit"),
        sum(when(col("transaction_type") == "CREDIT", col("amount")).otherwise(0)).alias("total_credit")
    ) \
    .withColumn("kpi_generated_time", current_timestamp())

In [0]:
fraud_df = df.withColumn(
    "fraud_flag",
    when(col("amount") > 150000, "HIGH_RISK")
    .when(col("amount") > 80000, "MEDIUM_RISK")
    .otherwise("LOW_RISK")
)

In [0]:
fraud_df = fraud_df.withColumn(
    "fraud_reason",
    when(col("amount") > 150000, "High transaction amount")
    .when(col("amount") > 80000, "Unusual transaction size")
    .otherwise("Normal behavior")
)

In [0]:
customer_df = df.groupBy("account_id") \
    .agg(
        count("*").alias("txn_count"),
        sum("amount").alias("total_spent")
    ) \
    .withColumn(
        "customer_segment",
        when(col("total_spent") > 500000, "HIGH_VALUE")
        .when(col("total_spent") > 100000, "MEDIUM_VALUE")
        .otherwise("LOW_VALUE")
    ) \
    .withColumn("last_updated", current_timestamp())

In [0]:
from delta.tables import DeltaTable


def upsert_table(df, table_name, key):

    if spark.catalog.tableExists(table_name):

        delta_table = DeltaTable.forName(spark, table_name)

        delta_table.alias("target").merge(
            df.alias("source"),
            f"target.{key} = source.{key}"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()

        print(f"Upsert completed: {table_name}")

    else:
        df.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(table_name)

        print(f"Created table: {table_name}")

In [0]:
upsert_table(kpi_df, KPI_TABLE, "location")
upsert_table(fraud_df, FRAUD_TABLE, "txn_id")
upsert_table(customer_df, CUSTOMER_TABLE, "account_id")

In [0]:
print("🔹 Gold Metrics")

print("KPI count:", kpi_df.count())
print("Fraud count:", fraud_df.count())
print("Customer count:", customer_df.count())

fraud_df.groupBy("fraud_flag").count().show()

In [0]:
%sql
OPTIMIZE banking.banking.gold_kpi_daily_transactions;
OPTIMIZE banking.banking.gold_fraud_transactions;
OPTIMIZE banking.banking.gold_customer_summary;